# S45_04 — PEFT and LoRA

**PEFT** (Parameter-Efficient Fine-Tuning) is a HuggingFace library that lets you fine-tune large models by updating only a tiny fraction of parameters — avoiding the cost of full fine-tuning while approaching its quality.

**LoRA** (Low-Rank Adaptation, Hu et al. 2021) is the most popular PEFT method. It freezes the original model weights and injects small trainable rank-decomposition matrices into selected layers.

## How LoRA works

For a weight matrix `W` (d × k), LoRA adds: `W' = W + B·A`  
where `A` is (r × k) and `B` is (d × r) with rank `r << min(d,k)`.  

- Only A and B are trained (~0.1–1% of total parameters)
- Original W is frozen — the base model stays intact
- Multiple LoRA adapters can be swapped without reloading the base model

In [ ]:
# pip install peft transformers accelerate
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load base model
model_name = 'bert-base-uncased'
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# LoRA config
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,   # sequence classification
    r=8,                           # rank — higher = more capacity but more params
    lora_alpha=16,                 # scaling factor (usually 2×r)
    target_modules=['query', 'value'],  # which weight matrices to adapt
    lora_dropout=0.1,
    bias='none',
)

# Wrap model with LoRA
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
# trainable params: ~296k / 109M  (≈0.27%)

## Training a LoRA adapter with the Trainer API

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tiny dataset for demo
ds = load_dataset('glue', 'sst2')
tokenized = ds.map(
    lambda x: tokenizer(x['sentence'], truncation=True, max_length=128),
    batched=True,
).rename_column('label', 'labels')
tokenized = tokenized.remove_columns(['sentence', 'idx'])
tokenized.set_format('torch')

# Training arguments — tiny for demo
training_args = TrainingArguments(
    output_dir='./lora-bert-sst2',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-4,           # LoRA typically needs higher LR than full fine-tune
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized['train'].select(range(2000)),  # small for demo
    eval_dataset=tokenized['validation'],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

# trainer.train()  # uncomment to actually train

## Saving and loading adapters

In [ ]:
# After training:
# peft_model.save_pretrained('./lora-adapter')   # saves ONLY the adapter weights (~4 MB)

# Load adapter onto fresh base model
from peft import PeftModel

# base = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
# loaded = PeftModel.from_pretrained(base, './lora-adapter')

# Merge adapter weights into base model for faster inference (optional)
# merged = loaded.merge_and_unload()  # returns a regular model with W + B·A merged

print('Adapter workflow:')
print('  save_pretrained()  → stores only adapter delta (~MB)')
print('  PeftModel.from_pretrained()  → load adapter onto any compatible base')
print('  merge_and_unload()  → bake adapter into weights for deployment')

## LoRA for causal LM (generative models)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Config for decoder-only generative models (GPT-style)
causal_lm_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    # For LLMs, target the attention projections and sometimes the MLP
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
)

print('For full LLM fine-tuning (e.g. Llama), combine LoRA with:')
print('  - QLoRA: 4-bit base model + LoRA adapters (see S49_02_lora_qlora.ipynb)')
print('  - Unsloth: 2–3× faster QLoRA training (see S49_05_finetuning_with_unsloth.ipynb)')

## Other PEFT methods

| Method | How it works | When to use |
|--------|-------------|-------------|
| **LoRA** | Low-rank weight delta | General purpose — default choice |
| **QLoRA** | LoRA + 4-bit base model | GPU memory limited (<24GB) |
| **Prompt tuning** | Learnable soft prompt tokens | Tiny adapter; weaker than LoRA |
| **Prefix tuning** | Learnable KV prefixes per layer | Sequence-to-sequence tasks |
| **IA³** | Scale keys/values/FFN by learned vectors | Very few parameters; fast |
| **Adapters** | Small FFN modules between layers | Older; LoRA usually better |

Next: [S45_05_accelerate.ipynb](./S45_05_accelerate.ipynb)